# tm_score-only production pipeline -- max_peaks=100 pre-NMS cap (K = 10, 20, 30)

**Updated from the previous version of this notebook**: precision is now reported only at
K = 10, 20, 30 (no 50, no 100). The pipeline and the two branches (`baseline`: `max_peaks`
never binds; `variant`: `max_peaks=100` applied before NMS) are unchanged from the previous
run -- see `z_floor_tightening_variant.ipynb` in this same directory for why a fixed pre-NMS
candidate-count cap and a per-ROI dynamically-solved z-floor targeting that same count are
mathematically the same operation; that notebook's `dynamic_z` branch cross-checks directly
against this one's mechanism.

**Everything else is held fixed at production's accepted defaults** (D8_TEMPLATE_ANCHOR.md,
current as of 2026-09-10): `hematoxylin_od` channel, `TM_CCOEFF` (unnormalized), single-scale/
single-angle/no-flip augmentation, `peak_min_distance=7`, `self_hit_radius=5.0`, NMS radius =
match radius = 7.5 um, same 14 ROIs, same one seed per ROI, same `tightened_template_box` seed
refinement, one `matchTemplate` pass per ROI reused by both branches.

**Sequential execution matters for the timing half of this question.** This notebook must not
be executed concurrently with its sibling ablation notebook.


In [1]:
import gc
import time
import sys

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Config -- identical to production_seed_precision_at_k_chromatin_half_pix_fix.ipynb's cell 1
# (D8_TEMPLATE_ANCHOR.md, current production as of 2026-09-10), and to
# latency_profiling/tm_score_latency_profile.ipynb, which reuses that exact config verbatim.
# Nothing here changes except the ablation block below.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

# ---------------------------------------------------------------------------------------
# The ablation. `extract_peaks`'s `max_peaks` argument truncates the local-maxima pool to
# its N best-scoring entries *before* NMS ever runs. Production sets it to 2,000,000 (never
# binds); this notebook caps it to 100.
BASELINE_DEEP_FLOOR_Z = -1.5      # unchanged in this notebook
VARIANT_DEEP_FLOOR_Z = -1.5       # unchanged in this notebook
BASELINE_MAX_PEAKS = 2_000_000    # production default -- never binds
VARIANT_MAX_PEAKS = 100
VARIANT_LABEL = 'max_peaks=100'

BUDGETS_FOR_EVAL = (10, 20, 30)    # the only budgets this run reports -- no 50, no 100
BUDGET = max(BUDGETS_FOR_EVAL)     # 30 -- stage 6's rank-and-truncate target

REFERENCE_RAW_CSV = '../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv'

print(f'BUDGETS_FOR_EVAL={BUDGETS_FOR_EVAL}, NMS radius = match radius = {NMS_RADIUS_UM} um, '
      f'channel={CHANNEL}')


BUDGETS_FOR_EVAL=(10, 20, 30), NMS radius = match radius = 7.5 um, channel=hematoxylin_od


## Helpers

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream."""
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')


14 ROIs on disk in ../images/extra_valid/


## Per-ROI worker

In [3]:
def run_roi(fn, image_id, domain, anns):
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    t_setup = time.perf_counter() - t0

    # ============= PIPELINE stages 1-3 (shared by both branches; timed once) ===========
    stages_shared = {}
    t_outer0 = time.perf_counter()

    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages_shared['t1_refine_seed_box_s'] = time.perf_counter() - t1

    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages_shared['t2_patch_template_build_s'] = time.perf_counter() - t2

    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages_shared['t3_template_matching_s'] = time.perf_counter() - t3

    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)

    # ========== branch: extraction (4) + NMS/self-hit (5) + rank-to-BUDGET (6) =========
    def run_branch(branch_name, deep_floor_z, max_peaks):
        b = {}
        t4 = time.perf_counter()
        cut = med + deep_floor_z * mad
        centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, max_peaks)
        b['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4
        n_peaks = len(centers)

        t5 = time.perf_counter()
        c, s = suppress(centers, scores, nms_radius, tpl_xy)
        b['t5_nms_selfhit_s'] = time.perf_counter() - t5
        assert bool(np.all(np.diff(s) <= 0)), \
            f'{fn}/{branch_name}: post-NMS pool is not score-descending'

        t6 = time.perf_counter()
        pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})
        top = pool.sort_values('score', ascending=False, na_position='last',
                               kind='mergesort').head(BUDGET)
        b['t6_rank_top_s'] = time.perf_counter() - t6

        d_seed = (np.hypot(pool['cx'] - tpl_xy[0], pool['cy'] - tpl_xy[1])
                 if len(pool) else np.array([]))
        n_near_seed = int((d_seed <= match_radius).sum())

        arm = cp.Arm('tm_score', (lambda d=pool: d), rank_key='score', seeded=True,
                     z=deep_floor_z, z_dependent=True, nms_radius=nms_radius,
                     caps=(max_peaks,), coverage_key=None)
        ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
                  base_size=base_size, branch=branch_name, deep_floor_z=deep_floor_z,
                  max_peaks=max_peaks, n_retries=n_retries,
                  map_median=round(float(med), 5), mad_scale=round(float(mad), 5))
        checks = []
        ev_out = cp.evaluate_arms([arm], gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                                  budgets=BUDGETS_FOR_EVAL, context=ctx, checks=checks)
        checks.append(dict(check='seed_annulus_empty', label=f'{fn}/{branch_name}',
                           n_near_seed=n_near_seed, passed=bool(n_near_seed == 0)))

        return dict(stages=b, n_peaks=n_peaks, n_detections=len(pool), ev_out=ev_out,
                   checks=checks, cut=cut)

    baseline = run_branch('baseline', BASELINE_DEEP_FLOOR_Z, BASELINE_MAX_PEAKS)
    variant = run_branch('variant', VARIANT_DEEP_FLOOR_Z, VARIANT_MAX_PEAKS)

    t_outer_total = time.perf_counter() - t_outer0
    gc.enable()

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch
    gc.collect()

    timing_row = dict(
        file_name=fn, tumor_type=domain, base_size=base_size, n_retries=n_retries,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        t_setup_s=round(t_setup, 5),
        t1_refine_seed_box_s=round(stages_shared['t1_refine_seed_box_s'], 5),
        t2_patch_template_build_s=round(stages_shared['t2_patch_template_build_s'], 5),
        t3_template_matching_s=round(stages_shared['t3_template_matching_s'], 5),
        n_peaks_baseline=baseline['n_peaks'], n_detections_baseline=baseline['n_detections'],
        t4_baseline_s=round(baseline['stages']['t4_threshold_peak_extraction_s'], 5),
        t5_baseline_s=round(baseline['stages']['t5_nms_selfhit_s'], 5),
        t6_baseline_s=round(baseline['stages']['t6_rank_top_s'], 5),
        n_peaks_variant=variant['n_peaks'], n_detections_variant=variant['n_detections'],
        t4_variant_s=round(variant['stages']['t4_threshold_peak_extraction_s'], 5),
        t5_variant_s=round(variant['stages']['t5_nms_selfhit_s'], 5),
        t6_variant_s=round(variant['stages']['t6_rank_top_s'], 5),
        t_outer_total_s=round(t_outer_total, 5),
    )

    print(f"[{fn}] {domain:32s} base={base_size:2d} "
          f"n_det base/var={baseline['n_detections']:6d}/{variant['n_detections']:6d}  "
          f"t4 base/var={baseline['stages']['t4_threshold_peak_extraction_s']*1000:6.1f}/"
          f"{variant['stages']['t4_threshold_peak_extraction_s']*1000:6.1f}ms  "
          f"t5 base/var={baseline['stages']['t5_nms_selfhit_s']*1000:6.1f}/"
          f"{variant['stages']['t5_nms_selfhit_s']*1000:6.1f}ms", flush=True)

    return timing_row, baseline['ev_out'], variant['ev_out'], baseline['checks'] + variant['checks']


## Run -- all 14 ROIs

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

timing_rows, baseline_evs, variant_evs, all_checks = [], [], [], []
t_run = time.time()
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    trow, b_ev, v_ev, checks = run_roi(fn, image_id, domain, annotations)
    timing_rows.append(trow)
    baseline_evs.append(b_ev)
    variant_evs.append(v_ev)
    all_checks.extend(checks)
    gc.collect()

TIMING = pd.DataFrame(timing_rows).set_index('file_name')
BASELINE_RAW = pd.concat(baseline_evs, ignore_index=True)
VARIANT_RAW = pd.concat(variant_evs, ignore_index=True)
CHECKS = pd.DataFrame(all_checks)

print(f'\n{len(files)} ROIs timed in {time.time() - t_run:.0f}s')
print(f'checks passed: {int(CHECKS["passed"].sum())}/{len(CHECKS)}')
assert CHECKS['passed'].all(), 'a pipeline-invariant check failed -- see CHECKS above'
CHECKS['check'].value_counts()


[013.tiff] human breast cancer              base=31 n_det base/var= 17724/    97  t4 base/var= 268.7/ 276.7ms  t5 base/var= 264.5/   0.7ms


[094.tiff] human breast cancer              base=25 n_det base/var= 18628/    96  t4 base/var= 255.9/ 256.5ms  t5 base/var= 292.0/   0.6ms


[201.tiff] canine lung cancer               base=51 n_det base/var= 15848/    98  t4 base/var= 211.1/ 213.1ms  t5 base/var= 125.9/   0.6ms


[233.tiff] canine lung cancer               base=25 n_det base/var= 17449/    97  t4 base/var= 215.4/ 210.0ms  t5 base/var= 206.2/   0.7ms


[245.tiff] canine lymphosarcoma             base=47 n_det base/var= 17678/    89  t4 base/var= 215.4/ 221.7ms  t5 base/var= 260.8/   0.6ms


...

[460.tiff] canine soft tissue sarcoma       base=47 n_det base/var= 15698/    99  t4 base/var= 313.7/ 314.8ms  t5 base/var= 177.7/   0.9ms


[529.tiff] human melanoma                   base=37 n_det base/var= 16620/    97  t4 base/var= 382.2/ 603.6ms  t5 base/var= 295.5/   0.9ms


[548.tiff] human melanoma                   base=29 n_det base/var= 17839/    98  t4 base/var= 394.5/ 401.9ms  t5 base/var= 390.1/   1.5ms



14 ROIs timed in 82s
checks passed: 84/84


check
no_cap                28
nms_radius            28
seed_annulus_empty    28
Name: count, dtype: int64

## Verification -- baseline branch reproduces committed production output

If this fails, nothing below can be trusted.

In [5]:
oracle = pd.read_csv(REFERENCE_RAW_CSV)
oracle_tm = oracle[oracle['arm'] == 'tm_score'].copy()

oracle_roi = (oracle_tm.drop_duplicates('file_name')
             .set_index('file_name')[['seed_ann_id', 'base_size', 'n_detections',
                                        'map_median', 'mad_scale']])
this_roi = TIMING[['seed_ann_id', 'base_size', 'n_detections_baseline', 'map_median',
                   'mad_scale']].rename(columns={'n_detections_baseline': 'n_detections'})

cmp = this_roi.join(oracle_roi, lsuffix='_this', rsuffix='_oracle')
mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp[f'{col}_this'].astype(int) != cmp[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp[f'{col}_this'], cmp[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))

# tp_at_K cross-check for every budget this run actually reports (10, 20, 30): confirms the
# ranking + bucketing + precision computation (not just the search + NMS) reproduces
# production exactly, at every budget this notebook cares about.
oracle_tp = oracle_tm.pivot_table(index='file_name', values='tp_at_budget', columns='budget')
this_baseline_tp = BASELINE_RAW.pivot_table(index='file_name', values='tp_at_budget', columns='budget')
for k in BUDGETS_FOR_EVAL:
    this_k = this_baseline_tp[k]
    oracle_k = oracle_tp.loc[this_k.index, k]
    bad = this_k.astype(int) != oracle_k.astype(int)
    if bad.any():
        mismatches.append((f'tp_at_{k}', this_k.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs. committed production reference -- baseline branch diverged:')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp)
    raise AssertionError('baseline branch does not reproduce the accepted production pipeline')

print(f"All {len(cmp)} ROIs' baseline branch matches {REFERENCE_RAW_CSV} exactly on "
      f"seed_ann_id / base_size / n_detections / map_median / mad_scale / "
      f"tp_at_{{{', '.join(str(k) for k in BUDGETS_FOR_EVAL)}}} (tm_score) -- this notebook's "
      f"harness reproduces production exactly before any variant is applied.")


All 14 ROIs' baseline branch matches ../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv exactly on seed_ann_id / base_size / n_detections / map_median / mad_scale / tp_at_{10, 20, 30} (tm_score) -- this notebook's harness reproduces production exactly before any variant is applied.


## Table 1 -- per-ROI stage timing, baseline vs. variant (ms)

In [6]:
STAGE_COLS_SHARED = ['t1_refine_seed_box_s', 't2_patch_template_build_s',
                    't3_template_matching_s']
BRANCH_COLS = ['t4_baseline_s', 't5_baseline_s', 't6_baseline_s',
              't4_variant_s', 't5_variant_s', 't6_variant_s']
ALL_TIME_COLS = ['t_setup_s'] + STAGE_COLS_SHARED + BRANCH_COLS + ['t_outer_total_s']

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(1)

display_cols = ['tumor_type', 'base_size', 'n_detections_baseline', 'n_detections_variant',
               't_setup_ms', 't1_refine_seed_box_ms', 't2_patch_template_build_ms',
               't3_template_matching_ms', 't4_baseline_ms', 't4_variant_ms',
               't5_baseline_ms', 't5_variant_ms', 't6_baseline_ms', 't6_variant_ms',
               't_outer_total_ms']
TIMING_MS.to_csv('max_peaks_100_timing.csv')
print(f'-> max_peaks_100_timing.csv')
TIMING_MS[display_cols]


-> max_peaks_100_timing.csv


,tumor_type,base_size,n_detections_baseline,n_detections_variant,t_setup_ms,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_baseline_ms,t4_variant_ms,t5_baseline_ms,t5_variant_ms,t6_baseline_ms,t6_variant_ms,t_outer_total_ms
file_name,,,,,,,,,,,,,,,
013.tiff,human breast cancer,31,17724,97,4285.4,1.0,0.0,1558.3,268.7,276.7,264.5,0.7,2.6,0.5,2710.4
094.tiff,human breast cancer,25,18628,96,2738.0,0.9,0.0,919.5,255.9,256.5,292.0,0.6,1.1,0.5,2069.8
201.tiff,canine lung cancer,51,15848,98,2702.5,1.3,0.0,1028.0,211.1,213.2,125.9,0.6,0.6,0.4,1841.8
233.tiff,canine lung cancer,25,17449,97,2136.4,1.2,0.0,810.8,215.4,210.0,206.2,0.6,0.6,0.4,1711.4
245.tiff,canine lymphosarcoma,47,17678,89,2029.5,0.8,0.0,939.7,215.4,221.7,260.8,0.6,0.6,0.5,1927.4
246.tiff,canine lymphosarcoma,41,18013,96,2170.6,1.4,0.0,925.0,221.8,214.6,193.1,0.7,0.7,0.4,1854.8
300.tiff,canine cutaneous mast cell tumor,45,17940,98,2744.4,1.3,0.0,986.2,203.0,205.0,156.1,0.6,0.9,0.4,1872.7
301.tiff,canine cutaneous mast cell tumor,41,17710,98,2359.7,1.2,0.0,989.0,218.8,227.5,185.4,0.7,0.7,0.5,1967.3
402.tiff,human neuroendocrine tumor,29,17532,98,3258.6,1.1,0.0,1224.7,318.8,309.4,337.7,0.8,1.3,0.5,2658.5


In [7]:
stage_pairs = [('t4_baseline_ms', 't4_variant_ms', 'stage 4 (threshold + peak extraction)'),
              ('t5_baseline_ms', 't5_variant_ms', 'stage 5 (NMS + self-hit)'),
              ('t6_baseline_ms', 't6_variant_ms', 'stage 6 (rank + top-K)')]

print('Shared stages (identical between branches by construction -- one matchTemplate pass reused):')
print(f'  t1 mean={TIMING_MS["t1_refine_seed_box_ms"].mean():.2f}ms  '
      f't2 mean={TIMING_MS["t2_patch_template_build_ms"].mean():.2f}ms  '
      f't3 mean={TIMING_MS["t3_template_matching_ms"].mean():.2f}ms')
print()
for base_col, var_col, label in stage_pairs:
    b_mean, v_mean = TIMING_MS[base_col].mean(), TIMING_MS[var_col].mean()
    delta = v_mean - b_mean
    pct = delta / b_mean * 100 if b_mean else float('nan')
    print(f'{label:38s}  baseline={b_mean:7.2f}ms  variant={v_mean:7.2f}ms  '
          f'delta={delta:+7.2f}ms ({pct:+.1f}%)')

total_base = TIMING_MS[['t1_refine_seed_box_ms', 't2_patch_template_build_ms',
                        't3_template_matching_ms', 't4_baseline_ms', 't5_baseline_ms',
                        't6_baseline_ms']].sum(axis=1)
total_var = TIMING_MS[['t1_refine_seed_box_ms', 't2_patch_template_build_ms',
                       't3_template_matching_ms', 't4_variant_ms', 't5_variant_ms',
                       't6_variant_ms']].sum(axis=1)
d_total = total_var.mean() - total_base.mean()
print(f'\nFull pipeline (stages 1-6) mean: baseline={total_base.mean():.1f}ms  '
      f'variant={total_var.mean():.1f}ms  delta={d_total:+.1f}ms '
      f'({d_total / total_base.mean() * 100:+.1f}%)')


Shared stages (identical between branches by construction -- one matchTemplate pass reused):
  t1 mean=1.23ms  t2 mean=0.00ms  t3 mean=1238.30ms

stage 4 (threshold + peak extraction)   baseline= 275.21ms  variant= 292.89ms  delta= +17.68ms (+6.4%)
stage 5 (NMS + self-hit)                baseline= 239.29ms  variant=   0.80ms  delta=-238.49ms (-99.7%)
stage 6 (rank + top-K)                  baseline=   1.07ms  variant=   0.52ms  delta=  -0.55ms (-51.3%)

Full pipeline (stages 1-6) mean: baseline=1755.1ms  variant=1533.7ms  delta=-221.4ms (-12.6%)


## Table 2 -- precision@{10,20,30}, baseline vs. variant

Long format: one row per (ROI, branch, budget). `budget_delivered` is reported explicitly
rather than assumed to equal K.

In [8]:
BASELINE_RAW['branch'] = 'baseline'
VARIANT_RAW['branch'] = 'variant'
ALL_RAW = pd.concat([BASELINE_RAW, VARIANT_RAW], ignore_index=True)
ALL_RAW['precision_at_budget'] = (ALL_RAW['tp_at_budget']
                                  / ALL_RAW['budget_delivered'].replace(0, np.nan))

PRECISION_LONG = ALL_RAW[['file_name', 'tumor_type', 'branch', 'budget', 'n_detections',
                          'n_gt_mitotic', 'budget_delivered', 'tp_at_budget',
                          'precision_at_budget', 'recall_at_budget']].copy()
PRECISION_LONG = PRECISION_LONG.sort_values(['file_name', 'budget', 'branch']).reset_index(drop=True)
PRECISION_LONG.to_csv('max_peaks_100_precision.csv', index=False)
print(f'-> max_peaks_100_precision.csv  ({len(PRECISION_LONG)} rows = 14 ROIs x 2 branches x '
      f'{len(BUDGETS_FOR_EVAL)} budgets)')

for branch in ['baseline', 'variant']:
    sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == BUDGET)]
    n_starved = int((sub['budget_delivered'] < BUDGET).sum())
    print(f'{branch:10s}: delivers fewer than BUDGET={BUDGET} candidates on '
          f'{n_starved}/{len(sub)} ROIs at the largest reported budget')

PRECISION_LONG.round(4)


-> max_peaks_100_precision.csv  (84 rows = 14 ROIs x 2 branches x 3 budgets)
baseline  : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget
variant   : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget


,file_name,tumor_type,branch,budget,n_detections,n_gt_mitotic,budget_delivered,tp_at_budget,precision_at_budget,recall_at_budget
0,013.tiff,human breast cancer,baseline,10,17724,17,10,3,0.3000,0.1765
1,013.tiff,human breast cancer,variant,10,97,17,10,3,0.3000,0.1765
2,013.tiff,human breast cancer,baseline,20,17724,17,20,4,0.2000,0.2353
3,013.tiff,human breast cancer,variant,20,97,17,20,4,0.2000,0.2353
4,013.tiff,human breast cancer,baseline,30,17724,17,30,6,0.2000,0.3529
5,013.tiff,human breast cancer,variant,30,97,17,30,6,0.2000,0.3529
6,094.tiff,human breast cancer,baseline,10,18628,81,10,4,0.4000,0.0494
7,094.tiff,human breast cancer,variant,10,96,81,10,4,0.4000,0.0494
8,094.tiff,human breast cancer,baseline,20,18628,81,20,10,0.5000,0.1235
9,094.tiff,human breast cancer,variant,20,96,81,20,10,0.5000,0.1235


## Table 2b -- precision@K pivoted for readability

In [9]:
PRECISION_PIVOT = PRECISION_LONG.pivot_table(
    index=['tumor_type', 'file_name'], columns=['budget', 'branch'], values='precision_at_budget'
)
PRECISION_PIVOT.round(4)


budget                                           10               20               30        
branch                                     baseline variant baseline variant baseline variant
tumor_type                       file_name                                                   
canine cutaneous mast cell tumor 300.tiff       0.7     0.7     0.70    0.70   0.6667  0.6667
                                 301.tiff       0.9     0.9     0.90    0.90   0.8000  0.8000
canine lung cancer               201.tiff       0.3     0.3     0.25    0.25   0.2000  0.2000
                                 233.tiff       0.3     0.3     0.25    0.25   0.2667  0.2667
canine lymphosarcoma             245.tiff       0.1     0.1     0.10    0.10   0.1000  0.1000
                                 246.tiff       1.0     1.0     0.90    0.90   0.8000  0.8000
canine soft tissue sarcoma       459.tiff       0.6     0.6     0.55    0.55   0.6333  0.6333
                                 460.tiff       0.7     0.7     0.50    0.50   0.4000  0.4000
human breast cancer              013.tiff       0.3     0.3     0.20    0.20   0.2000  0.2000
                                 094.tiff       0.4     0.4     0.50    0.50   0.5333  0.5333
human melanoma                   529.tiff       0.3     0.3     0.20    0.20   0.1333  0.1333
                                 548.tiff       0.5     0.5     0.50    0.50   0.5333  0.5333
human neuroendocrine tumor       402.tiff       0.4     0.4     0.45    0.45   0.4667  0.4667
                                 403.tiff       0.5     0.5     0.35    0.35   0.2667  0.2667

## Summary -- pooled precision and win counts, by budget

In [10]:
print(f'Pooled precision (sum tp / sum delivered) across all 14 ROIs, by branch and budget:')
for k in BUDGETS_FOR_EVAL:
    print(f'  K={k}:')
    for branch in ['baseline', 'variant']:
        sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        pooled = sub['tp_at_budget'].sum() / sub['budget_delivered'].sum()
        print(f'    {branch:10s}: {pooled:.4f}  ({int(sub["tp_at_budget"].sum())} tp / '
              f'{int(sub["budget_delivered"].sum())} delivered)')

print()
for k in BUDGETS_FOR_EVAL:
    base_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'baseline') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    var_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'variant') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    d_var = (var_k['precision_at_budget'] - base_k['precision_at_budget'])
    r_var = (var_k['recall_at_budget'] < base_k['recall_at_budget']).sum()
    print(f'K={k}: variant precision delta: {int((d_var > 0).sum())} up / '
          f'{int((d_var < 0).sum())} down / {int((d_var == 0).sum())} unchanged of 14 ROIs; '
          f'recall lower on {int(r_var)}/14')


Pooled precision (sum tp / sum delivered) across all 14 ROIs, by branch and budget:
  K=10:
    baseline  : 0.5000  (70 tp / 140 delivered)
    variant   : 0.5000  (70 tp / 140 delivered)
  K=20:
    baseline  : 0.4536  (127 tp / 280 delivered)
    variant   : 0.4536  (127 tp / 280 delivered)
  K=30:
    baseline  : 0.4286  (180 tp / 420 delivered)
    variant   : 0.4286  (180 tp / 420 delivered)

K=10: variant precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14
K=20: variant precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14
K=30: variant precision delta: 0 up / 0 down / 14 unchanged of 14 ROIs; recall lower on 0/14


## Closing readout

In [11]:
print(f'{VARIANT_LABEL} vs. baseline, mean over {len(TIMING)} ROIs:')
print(f'  full pipeline (stages 1-6): {total_base.mean():.1f}ms -> {total_var.mean():.1f}ms '
      f'({d_total:+.1f}ms, {d_total / total_base.mean() * 100:+.1f}%)')
print()
print('Precision, pooled across 14 ROIs (baseline -> variant):')
for k in BUDGETS_FOR_EVAL:
    b = PRECISION_LONG[(PRECISION_LONG['branch'] == 'baseline') & (PRECISION_LONG['budget'] == k)]
    v = PRECISION_LONG[(PRECISION_LONG['branch'] == 'variant') & (PRECISION_LONG['budget'] == k)]
    pb = b['tp_at_budget'].sum() / b['budget_delivered'].sum()
    pv = v['tp_at_budget'].sum() / v['budget_delivered'].sum()
    n_starved = int((v['budget_delivered'] < k).sum())
    print(f'  K={k}: {pb:.4f} -> {pv:.4f}  (under-delivers K={k} on {n_starved}/14 ROIs)')
print()
print(f'Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, '
      f'multi-seed sweep D5 itself sets as the bar for changing a production default.')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')


max_peaks=100 vs. baseline, mean over 14 ROIs:
  full pipeline (stages 1-6): 1755.1ms -> 1533.7ms (-221.4ms, -12.6%)

Precision, pooled across 14 ROIs (baseline -> variant):
  K=10: 0.5000 -> 0.5000  (under-delivers K=10 on 0/14 ROIs)
  K=20: 0.4536 -> 0.4536  (under-delivers K=20 on 0/14 ROIs)
  K=30: 0.4286 -> 0.4286  (under-delivers K=30 on 0/14 ROIs)

Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, multi-seed sweep D5 itself sets as the bar for changing a production default.

notebook ran in 83s
